#### Copyright 2018 Google LLC.

In [ ]:
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Cat vs. Dog Image Classification
## Exercise 1: Building a Convnet from Scratch
**_Estimated completion time: 20 minutes_**

In this exercise, we will build a classifier model from scratch that is able to distinguish dogs from cats. We will follow these steps:

1. Explore the example data
2. Build a small convnet from scratch to solve our classification problem
3. Evaluate training and validation accuracy

Let's go!

## Explore the Example Data

Let's start by downloading our example data, a .zip of 2,000 JPG pictures of cats and dogs, and extracting it locally in `/tmp`.

**NOTE:** The 2,000 images used in this exercise are excerpted from the ["Dogs vs. Cats" dataset](https://www.kaggle.com/c/dogs-vs-cats/data) available on Kaggle, which contains 25,000 images. Here, we use a subset of the full dataset to decrease training time for educational purposes.

In [ ]:
!wget --no-check-certificate \
    https://storage.googleapis.com/mledu-datasets/cats_and_dogs_filtered.zip \
    -O /tmp/cats_and_dogs_filtered.zip

In [ ]:
import os
import zipfile

local_zip = '/tmp/cats_and_dogs_filtered.zip'
zip_ref = zipfile.ZipFile(local_zip, 'r')
zip_ref.extractall('/tmp')
zip_ref.close()

The contents of the .zip are extracted to the base directory `/tmp/cats_and_dogs_filtered`, which contains `train` and `validation` subdirectories for the training and validation datasets (see the [Machine Learning Crash Course](https://developers.google.com/machine-learning/crash-course/validation/check-your-intuition) for a refresher on training, validation, and test sets), which in turn each contain `cats` and `dogs` subdirectories. Let's define each of these directories:

In [ ]:
base_dir = '/tmp/cats_and_dogs_filtered'
train_dir = os.path.join(base_dir, 'train')
validation_dir = os.path.join(base_dir, 'validation')

# Directory with our training cat pictures
train_cats_dir = os.path.join(train_dir, 'cats')

# Directory with our training dog pictures
train_dogs_dir = os.path.join(train_dir, 'dogs')

# Directory with our validation cat pictures
validation_cats_dir = os.path.join(validation_dir, 'cats')

# Directory with our validation dog pictures
validation_dogs_dir = os.path.join(validation_dir, 'dogs')

Now, let's see what the filenames look like in the `cats` and `dogs` `train` directories (file naming conventions are the same in the `validation` directory):

In [ ]:
train_cat_fnames = os.listdir(train_cats_dir)
print(train_cat_fnames[:10])

train_dog_fnames = os.listdir(train_dogs_dir)
train_dog_fnames.sort()
print(train_dog_fnames[:10])

Let's find out the total number of cat and dog images in the `train` and `validation` directories:

In [ ]:
print('total training cat images:', len(os.listdir(train_cats_dir)))
print('total training dog images:', len(os.listdir(train_dogs_dir)))
print('total validation cat images:', len(os.listdir(validation_cats_dir)))
print('total validation dog images:', len(os.listdir(validation_dogs_dir)))

For both cats and dogs, we have 1,000 training images and 500 test images.

Now let's take a look at a few pictures to get a better sense of what the cat and dog datasets look like. First, configure the matplot parameters:

In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Parameters for our graph; we'll output images in a 4x4 configuration
nrows = 4
ncols = 4

# Index for iterating over images
pic_index = 0

Now, display a batch of 8 cat and 8 dog pictures. You can rerun the cell to see a fresh batch each time:

In [ ]:
# Set up matplotlib fig, and size it to fit 4x4 pics
fig = plt.gcf()
fig.set_size_inches(ncols * 4, nrows * 4)

pic_index += 8
next_cat_pix = [os.path.join(train_cats_dir, fname)
                for fname in train_cat_fnames[pic_index-8:pic_index]]
next_dog_pix = [os.path.join(train_dogs_dir, fname)
                for fname in train_dog_fnames[pic_index-8:pic_index]]

for i, img_path in enumerate(next_cat_pix+next_dog_pix):
  # Set up subplot; subplot indices start at 1
  sp = plt.subplot(nrows, ncols, i + 1)
  sp.axis('Off') # Don't show axes (or gridlines)

  img = mpimg.imread(img_path)
  plt.imshow(img)

plt.show()


## Building a Small Convnet from Scratch to Get to 72% Accuracy

The images that will go into our convnet are 150x150 color images (in the next section on Data Preprocessing, we'll add handling to resize all the images to 150x150 before feeding them into the neural network).

Let's code up the architecture. We will stack 3 {convolution + relu + maxpooling} modules. Our convolutions operate on 3x3 windows and our maxpooling layers operate on 2x2 windows. Our first convolution extracts 16 filters, the following one extracts 32 filters, and the last one extracts 64 filters.

**NOTE**: This is a configuration that is widely used and known to work well for image classification. Also, since we have relatively few training examples (1,000), using just three convolutional modules keeps the model small, which lowers the risk of overfitting (which we'll explore in more depth in Exercise 2.)

In [ ]:
from tensorflow.keras import layers
from tensorflow.keras import Model

In [ ]:
# Our input feature map is 150x150x3: 150x150 for the image pixels, and 3 for
# the three color channels: R, G, and B
img_input = layers.Input(shape=(150, 150, 3))

# First convolution extracts 16 filters that are 3x3
# Convolution is followed by max-pooling layer with a 2x2 window
x = layers.Conv2D(16, 3, activation='relu')(img_input)
x = layers.MaxPooling2D(2)(x)

# Second convolution extracts 32 filters that are 3x3
# Convolution is followed by max-pooling layer with a 2x2 window
x = layers.Conv2D(32, 3, activation='relu')(x)
x = layers.MaxPooling2D(2)(x)

# Third convolution extracts 64 filters that are 3x3
# Convolution is followed by max-pooling layer with a 2x2 window
x = layers.Conv2D(64, 3, activation='relu')(x)
x = layers.MaxPooling2D(2)(x)

On top of it we stick two fully-connected layers. Because we are facing a two-class classification problem, i.e. a *binary classification problem*, we will end our network with a [*sigmoid* activation](https://wikipedia.org/wiki/Sigmoid_function), so that the output of our network will be a single scalar between 0 and 1, encoding the probability that the current image is class 1 (as opposed to class 0).

In [ ]:
# Flatten feature map to a 1-dim tensor so we can add fully connected layers
x = layers.Flatten()(x)

# Create a fully connected layer with ReLU activation and 512 hidden units
x = layers.Dense(512, activation='relu')(x)

# Create output layer with a single node and sigmoid activation
output = layers.Dense(1, activation='sigmoid')(x)

# Create model:
# input = input feature map
# output = input feature map + stacked convolution/maxpooling layers + fully
# connected layer + sigmoid output layer
model = Model(img_input, output)

Let's summarize the model architecture:

In [ ]:
model.summary()

The "output shape" column shows how the size of your feature map evolves in each successive layer. The convolution layers reduce the size of the feature maps by a bit due to padding, and each pooling layer halves the feature map.

Next, we'll configure the specifications for model training. We will train our model with the `binary_crossentropy` loss, because it's a binary classification problem and our final activation is a sigmoid. (For a refresher on loss metrics, see the [Machine Learning Crash Course](https://developers.google.com/machine-learning/crash-course/descending-into-ml/video-lecture).) We will use the `rmsprop` optimizer with a learning rate of `0.001`. During training, we will want to monitor classification accuracy.

**NOTE**: In this case, using the [RMSprop optimization algorithm](https://wikipedia.org/wiki/Stochastic_gradient_descent#RMSProp) is preferable to [stochastic gradient descent](https://developers.google.com/machine-learning/glossary/#SGD) (SGD), because RMSprop automates learning-rate tuning for us. (Other optimizers, such as [Adam](https://wikipedia.org/wiki/Stochastic_gradient_descent#Adam) and [Adagrad](https://developers.google.com/machine-learning/glossary/#AdaGrad), also automatically adapt the learning rate during training, and would work equally well here.)

In [ ]:
from tensorflow.keras.optimizers import RMSprop

model.compile(loss='binary_crossentropy',
              optimizer=RMSprop(lr=0.001),
              metrics=['acc'])

### Data Preprocessing

Let's set up data generators that will read pictures in our source folders, convert them to `float32` tensors, and feed them (with their labels) to our network. We'll have one generator for the training images and one for the validation images. Our generators will yield batches of 20 images of size 150x150 and their labels (binary).

As you may already know, data that goes into neural networks should usually be normalized in some way to make it more amenable to processing by the network. (It is uncommon to feed raw pixels into a convnet.) In our case, we will preprocess our images by normalizing the pixel values to be in the `[0, 1]` range (originally all values are in the `[0, 255]` range).

In Keras this can be done via the `keras.preprocessing.image.ImageDataGenerator` class using the `rescale` parameter. This `ImageDataGenerator` class allows you to instantiate generators of augmented image batches (and their labels) via `.flow(data, labels)` or `.flow_from_directory(directory)`. These generators can then be used with the Keras model methods that accept data generators as inputs: `fit_generator`, `evaluate_generator`, and `predict_generator`.

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# All images will be rescaled by 1./255
train_datagen = ImageDataGenerator(rescale=1./255)
val_datagen = ImageDataGenerator(rescale=1./255)

# Flow training images in batches of 20 using train_datagen generator
train_generator = train_datagen.flow_from_directory(
        train_dir,  # This is the source directory for training images
        target_size=(150, 150),  # All images will be resized to 150x150
        batch_size=20,
        # Since we use binary_crossentropy loss, we need binary labels
        class_mode='binary')

# Flow validation images in batches of 20 using val_datagen generator
validation_generator = val_datagen.flow_from_directory(
        validation_dir,
        target_size=(150, 150),
        batch_size=20,
        class_mode='binary')

### Training
Let's train on all 2,000 images available, for 15 epochs, and validate on all 1,000 validation images. (This may take a few minutes to run.)

In [ ]:
history = model.fit_generator(
      train_generator,
      steps_per_epoch=100,  # 2000 images = batch_size * steps
      epochs=15,
      validation_data=validation_generator,
      validation_steps=50,  # 1000 images = batch_size * steps
      verbose=2)

### Visualizing Intermediate Representations

To get a feel for what kind of features our convnet has learned, one fun thing to do is to visualize how an input gets transformed as it goes through the convnet.

Let's pick a random cat or dog image from the training set, and then generate a figure where each row is the output of a layer, and each image in the row is a specific filter in that output feature map. Rerun this cell to generate intermediate representations for a variety of training images.

In [ ]:
import numpy as np
import random
from tensorflow.keras.preprocessing.image import img_to_array, load_img

# Let's define a new Model that will take an image as input, and will output
# intermediate representations for all layers in the previous model after
# the first.
successive_outputs = [layer.output for layer in model.layers[1:]]
visualization_model = Model(img_input, successive_outputs)

# Let's prepare a random input image of a cat or dog from the training set.
cat_img_files = [os.path.join(train_cats_dir, f) for f in train_cat_fnames]
dog_img_files = [os.path.join(train_dogs_dir, f) for f in train_dog_fnames]
img_path = random.choice(cat_img_files + dog_img_files)

img = load_img(img_path, target_size=(150, 150))  # this is a PIL image
x = img_to_array(img)  # Numpy array with shape (150, 150, 3)
x = x.reshape((1,) + x.shape)  # Numpy array with shape (1, 150, 150, 3)

# Rescale by 1/255
x /= 255

# Let's run our image through our network, thus obtaining all
# intermediate representations for this image.
successive_feature_maps = visualization_model.predict(x)

# These are the names of the layers, so can have them as part of our plot
layer_names = [layer.name for layer in model.layers[1:]]

# Now let's display our representations
for layer_name, feature_map in zip(layer_names, successive_feature_maps):
  if len(feature_map.shape) == 4:
    # Just do this for the conv / maxpool layers, not the fully-connected layers
    n_features = feature_map.shape[-1]  # number of features in feature map
    # The feature map has shape (1, size, size, n_features)
    size = feature_map.shape[1]
    # We will tile our images in this matrix
    display_grid = np.zeros((size, size * n_features))
    for i in range(n_features):
      # Postprocess the feature to make it visually palatable
      x = feature_map[0, :, :, i]
      x -= x.mean()
      x /= x.std()
      x *= 64
      x += 128
      x = np.clip(x, 0, 255).astype('uint8')
      # We'll tile each filter into this big horizontal grid
      display_grid[:, i * size : (i + 1) * size] = x
    # Display the grid
    scale = 20. / n_features
    plt.figure(figsize=(scale * n_features, scale))
    plt.title(layer_name)
    plt.grid(False)
    plt.imshow(display_grid, aspect='auto', cmap='viridis')

As you can see we go from the raw pixels of the images to increasingly abstract and compact representations. The representations downstream start highlighting what the network pays attention to, and they show fewer and fewer features being "activated"; most are set to zero. This is called "sparsity." Representation sparsity is a key feature of deep learning.


These representations carry increasingly less information about the original pixels of the image, but increasingly refined information about the class of the image. You can think of a convnet (or a deep network in general) as an information distillation pipeline.

### Evaluating Accuracy and Loss for the Model

Let's plot the training/validation accuracy and loss as collected during training:

In [ ]:
# Retrieve a list of accuracy results on training and validation data
# sets for each training epoch
acc = history.history['acc']
val_acc = history.history['val_acc']

# Retrieve a list of list results on training and validation data
# sets for each training epoch
loss = history.history['loss']
val_loss = history.history['val_loss']

# Get number of epochs
epochs = range(len(acc))

# Plot training and validation accuracy per epoch
plt.plot(epochs, acc)
plt.plot(epochs, val_acc)
plt.title('Training and validation accuracy')

plt.figure()

# Plot training and validation loss per epoch
plt.plot(epochs, loss)
plt.plot(epochs, val_loss)
plt.title('Training and validation loss')

As you can see, we are **overfitting** like it's getting out of fashion. Our training accuracy (in blue) gets close to 100% (!) while our validation accuracy (in green) stalls as 70%. Our validation loss reaches its minimum after only five epochs.

Since we have a relatively small number of training examples (2000), overfitting should be our number one concern. Overfitting happens when a model exposed to too few examples learns patterns that do not generalize to new data, i.e. when the model starts using irrelevant features for making predictions. For instance, if you, as a human, only see three images of people who are lumberjacks, and three images of people who are sailors, and among them the only person wearing a cap is a lumberjack, you might start thinking that wearing a cap is a sign of being a lumberjack as opposed to a sailor. You would then make a pretty lousy lumberjack/sailor classifier.

Overfitting is the central problem in machine learning: given that we are fitting the parameters of our model to a given dataset, how can we make sure that the representations learned by the model will be applicable to data never seen before? How do we avoid learning things that are specific to the training data?

In the next exercise, we'll look at ways to prevent overfitting in the cat vs. dog classification model.

## Clean Up

Before running the next exercise, run the following cell to terminate the kernel and free memory resources:

In [ ]:
import os, signal
os.kill(os.getpid(), signal.SIGKILL)

## Introducción al Cuaderno: Clasificación de Imágenes de Perros y Gatos

Este cuaderno tiene como objetivo construir un modelo de clasificación desde cero capaz de distinguir entre imágenes de perros y gatos. Seguiremos los siguientes pasos:

1.  **Explorar los datos de ejemplo:** Descargar y entender la estructura del conjunto de datos.
2.  **Construir una pequeña red neuronal convolucional (Convnet):** Diseñar y definir la arquitectura del modelo.
3.  **Evaluar la precisión:** Entrenar el modelo y analizar su rendimiento en los conjuntos de entrenamiento y validación.

El código inicial establece las licencias y la configuración básica del entorno.

## Descarga y Extracción de Datos

Esta sección se encarga de obtener el conjunto de datos de imágenes. El conjunto de datos consiste en un archivo `.zip` que contiene 2,000 imágenes JPG de perros y gatos, extraídas de un conjunto de datos más grande de Kaggle.

**Propósito:** Acceder a las imágenes necesarias para el entrenamiento y la validación del modelo.

**Importancia:** Sin estos datos, el modelo no tendría nada que aprender.

**Relación:** Los archivos descargados y extraídos son la fuente de entrada para todo el proceso de preprocesamiento y entrenamiento del modelo.

In [6]:
# Descarga el archivo zip que contiene las imágenes de perros y gatos
# El comando !wget se usa para descargar archivos desde la web en un entorno de Colab.
!wget --no-check-certificate \
    https://storage.googleapis.com/mledu-datasets/cats_and_dogs_filtered.zip \
    -O /tmp/cats_and_dogs_filtered.zip

--2026-03-18 04:48:48--  https://storage.googleapis.com/mledu-datasets/cats_and_dogs_filtered.zip
Resolving storage.googleapis.com (storage.googleapis.com)... 74.125.137.207, 142.250.101.207, 142.250.141.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|74.125.137.207|:443... connected.
HTTP request sent, awaiting response... 403 Forbidden
2026-03-18 04:48:48 ERROR 403: Forbidden.



In [7]:
import os
import zipfile

# Define la ruta del archivo zip descargado
local_zip = '/tmp/cats_and_dogs_filtered.zip'

# Abre el archivo zip y extrae todo su contenido en el directorio /tmp
zip_ref = zipfile.ZipFile(local_zip, 'r')
zip_ref.extractall('/tmp')

# Cierra el archivo zip después de la extracción
zip_ref.close()

BadZipFile: File is not a zip file

## Configuración de Directorios y Rutas de Datos

Una vez que los datos han sido extraídos, es crucial organizar y definir las rutas a los diferentes subconjuntos de imágenes (entrenamiento y validación, y dentro de estos, perros y gatos). Esto facilita el acceso programático a las imágenes.

**Propósito:** Crear variables que apunten a las ubicaciones exactas de las imágenes de entrenamiento y validación para perros y gatos.

**Importancia:** Permite que los generadores de datos de Keras (que veremos más adelante) carguen las imágenes correctamente.

**Relación:** Estas rutas son utilizadas directamente por la función `flow_from_directory` del `ImageDataGenerator`.

In [3]:
import os

# Directorio base donde se extrajeron los datos
base_dir = '/tmp/cats_and_dogs_filtered'

# Directorios para el conjunto de entrenamiento y validación
train_dir = os.path.join(base_dir, 'train')
validation_dir = os.path.join(base_dir, 'validation')

# Directorios específicos para imágenes de gatos en entrenamiento y validación
train_cats_dir = os.path.join(train_dir, 'cats')
validation_cats_dir = os.path.join(validation_dir, 'cats')

# Directorios específicos para imágenes de perros en entrenamiento y validación
train_dogs_dir = os.path.join(train_dir, 'dogs')
validation_dogs_dir = os.path.join(validation_dir, 'dogs')

## Inspección del Contenido de los Directorios

Esta parte del código verifica el contenido de los directorios de datos para confirmar que las imágenes se extrajeron correctamente y para entender la cantidad de imágenes disponibles en cada categoría (perros y gatos) para los conjuntos de entrenamiento y validación.

**Propósito:** Contar el número de imágenes en cada subdirectorio y mostrar algunos nombres de archivo para verificar la estructura.

**Importancia:** Asegura que el conjunto de datos está presente y accesible, y da una idea del balance de clases.

**Relación:** La cuenta de imágenes es fundamental para configurar correctamente los `steps_per_epoch` y `validation_steps` en el entrenamiento del modelo.

In [1]:
import os

# Lista los primeros 10 nombres de archivo de gatos en el directorio de entrenamiento
train_cat_fnames = os.listdir(train_cats_dir)
print('Primeros 10 nombres de archivos de gatos de entrenamiento:', train_cat_fnames[:10])

# Lista los primeros 10 nombres de archivo de perros en el directorio de entrenamiento (ordenados)
train_dog_fnames = os.listdir(train_dogs_dir)
train_dog_fnames.sort()
print('Primeros 10 nombres de archivos de perros de entrenamiento:', train_dog_fnames[:10])

NameError: name 'train_cats_dir' is not defined

In [2]:
import os

# Imprime el número total de imágenes en cada categoría y conjunto
print('Total de imágenes de gatos de entrenamiento:', len(os.listdir(train_cats_dir)))
print('Total de imágenes de perros de entrenamiento:', len(os.listdir(train_dogs_dir)))
print('Total de imágenes de gatos de validación:', len(os.listdir(validation_cats_dir)))
print('Total de imágenes de perros de validación:', len(os.listdir(validation_dogs_dir)))

NameError: name 'train_cats_dir' is not defined

## Visualización de Imágenes de Ejemplo

Para obtener una mejor comprensión de los datos con los que estamos trabajando, esta sección configura `matplotlib` para mostrar un lote de imágenes de gatos y perros del conjunto de entrenamiento. Esto ayuda a visualizar la calidad, el formato y la variedad de las imágenes.

**Propósito:** Mostrar visualmente ejemplos de las imágenes de gatos y perros.

**Importancia:** Proporciona una idea de lo que el modelo "ve" y ayuda a identificar posibles problemas con los datos (por ejemplo, imágenes corruptas, fondos complejos).

**Relación:** Estas imágenes son el tipo de entrada que se procesará a través de la red neuronal.

In [3]:
# Configuración para que las gráficas se muestren en el cuaderno (modo inline)
%matplotlib inline

import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Parámetros para la visualización: una cuadrícula de 4x4 imágenes
nrows = 4
ncols = 4

# Índice para iterar sobre las imágenes (se usará más adelante)
pic_index = 0

In [4]:
import os
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Crea una figura de matplotlib y ajusta su tamaño
fig = plt.gcf()
fig.set_size_inches(ncols * 4, nrows * 4)

# Incrementa el índice de imágenes para mostrar un nuevo lote cada vez que se ejecuta la celda
pic_index += 8

# Obtiene las rutas de los siguientes 8 archivos de imágenes de gatos y perros
next_cat_pix = [os.path.join(train_cats_dir, fname)
                for fname in train_cat_fnames[pic_index-8:pic_index]]
next_dog_pix = [os.path.join(train_dogs_dir, fname)
                for fname in train_dog_fnames[pic_index-8:pic_index]]

# Itera a través de las imágenes y las muestra en la cuadrícula
for i, img_path in enumerate(next_cat_pix+next_dog_pix):
  # Configura un subplot para cada imagen
  sp = plt.subplot(nrows, ncols, i + 1)
  sp.axis('Off') # Desactiva los ejes

  img = mpimg.imread(img_path)
  plt.imshow(img)

plt.show() # Muestra la figura con todas las imágenes

NameError: name 'train_cat_fnames' is not defined

<Figure size 1600x1600 with 0 Axes>

## Construcción de una Pequeña Convnet desde Cero

Esta sección describe la arquitectura de la red neuronal convolucional (Convnet) que construiremos. El modelo está diseñado para clasificar imágenes de 150x150 píxeles a color. La arquitectura es una configuración común para la clasificación de imágenes y es lo suficientemente pequeña para evitar el sobreajuste con un conjunto de datos limitado.

**Propósito:** Definir las capas y la estructura de la red neuronal.

**Importancia:** La arquitectura del modelo es el "cerebro" que aprenderá a identificar patrones en las imágenes.

**Relación:** Las capas definidas aquí son los bloques de construcción de la función `Model` de Keras que se creará a continuación.

In [5]:
# Importa las capas necesarias de TensorFlow Keras
from tensorflow.keras import layers
from tensorflow.keras import Model

### Definición de las Capas Convolucionales y de Max-Pooling

Aquí se definen las capas convolucionales (Conv2D) y de agrupación máxima (MaxPooling2D) que forman el cuerpo de la red. Estas capas son fundamentales para extraer características de las imágenes.

-   **`layers.Input(shape=(150, 150, 3))`**: Define la forma de entrada de las imágenes (150x150 píxeles, 3 canales de color: RGB).
-   **`layers.Conv2D(filtros, tamaño_kernel, activation='relu')`**: Aplica filtros para detectar patrones en la imagen. `relu` es la función de activación.
-   **`layers.MaxPooling2D(tamaño_pooling)`**: Reduce las dimensiones espaciales de la salida de la convolución, manteniendo las características más importantes y reduciendo la complejidad computacional.

**Propósito:** Construir las capas que aprenderán características jerárquicas de las imágenes.

**Importancia:** Las capas convolucionales y de pooling son el núcleo de las Convnet, permitiendo al modelo entender las características visuales.

**Relación:** La salida de estas capas será la entrada para las capas completamente conectadas que tomarán la decisión final.

In [6]:
# Define la capa de entrada con la forma de las imágenes (150x150 píxeles, 3 canales de color)
img_input = layers.Input(shape=(150, 150, 3))

# Primer bloque convolucional: 16 filtros 3x3, activación ReLU, seguido de MaxPooling 2x2
x = layers.Conv2D(16, 3, activation='relu')(img_input)
x = layers.MaxPooling2D(2)(x)

# Segundo bloque convolucional: 32 filtros 3x3, activación ReLU, seguido de MaxPooling 2x2
x = layers.Conv2D(32, 3, activation='relu')(x)
x = layers.MaxPooling2D(2)(x)

# Tercer bloque convolucional: 64 filtros 3x3, activación ReLU, seguido de MaxPooling 2x2
x = layers.Conv2D(64, 3, activation='relu')(x)
x = layers.MaxPooling2D(2)(x)

### Definición de las Capas Completamente Conectadas y la Capa de Salida

Después de las capas convolucionales, se añaden capas completamente conectadas (Dense) para procesar las características extraídas y producir una predicción final. Dado que es un problema de clasificación binaria (perro o gato), la capa de salida usa una activación `sigmoid`.

-   **`layers.Flatten()(x)`**: Convierte el mapa de características 2D (salida de la última capa de pooling) en un vector 1D para alimentarlo a las capas `Dense`.
-   **`layers.Dense(512, activation='relu')(x)`**: Una capa oculta con 512 neuronas y activación `relu`.
-   **`layers.Dense(1, activation='sigmoid')(x)`**: La capa de salida final. Una sola neurona con activación `sigmoid` produce un valor entre 0 y 1, que representa la probabilidad de que la imagen pertenezca a la clase '1' (por ejemplo, 'perro').
-   **`model = Model(img_input, output)`**: Combina las capas de entrada y salida para crear el modelo completo de Keras.

**Propósito:** Traducir las características de alto nivel en una predicción de clase.

**Importancia:** Estas capas toman la decisión final de clasificación.

**Relación:** `Flatten` conecta las características aprendidas por la CNN con las capas densas, y la capa de salida proporciona la probabilidad final que se evaluará.

In [7]:
# Aplana el mapa de características (la salida de la última capa de MaxPooling) a un tensor 1D
x = layers.Flatten()(x)

# Crea una capa completamente conectada (Dense) con 512 unidades y activación ReLU
x = layers.Dense(512, activation='relu')(x)

# Crea la capa de salida con una sola unidad y activación Sigmoid para clasificación binaria
output = layers.Dense(1, activation='sigmoid')(x)

# Crea el modelo combinando la entrada y la salida definidas
model = Model(img_input, output)

## Resumen del Modelo

El método `model.summary()` imprime un resumen textual de la arquitectura del modelo. Esto incluye el nombre y tipo de cada capa, la forma de su salida (`Output Shape`) y el número de parámetros entrenables (`Param #`).

**Propósito:** Obtener una visión general rápida y detallada de la estructura del modelo.

**Importancia:** Ayuda a depurar la arquitectura, verificar las dimensiones de las capas y entender el número total de parámetros que el modelo necesita aprender.

**Relación:** Muestra la culminación de la definición de las capas anteriores, presentando cómo se conectan y evolucionan las dimensiones de los datos a través de la red.

In [11]:
# Muestra un resumen de la arquitectura del modelo, incluyendo capas, formas de salida y número de parámetros
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 150, 150, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 148, 148, 16)   │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 74, 74, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 72, 72, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 36, 36, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 34, 34, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 17, 17, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 18496)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │     9,470,464 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           513 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,494,561 (36.22 MB)

 Trainable params: 9,494,561 (36.22 MB)

 Non-trainable params: 0 (0.00 B)

## Compilación del Modelo

Antes de entrenar el modelo, es necesario compilarlo. Esto implica definir la función de pérdida (que el modelo intentará minimizar), el optimizador (cómo se ajustarán los pesos) y las métricas (qué se medirá durante el entrenamiento).

-   **`loss='binary_crossentropy'`**: Se elige esta función de pérdida porque es un problema de clasificación binaria y la capa de salida usa activación `sigmoid`.
-   **`optimizer=RMSprop(lr=0.001)`**: El optimizador RMSprop se utiliza para ajustar los pesos del modelo. Es preferible a SGD en este caso porque automatiza la sintonización de la tasa de aprendizaje.
-   **`metrics=['acc']`**: Se monitoreará la precisión (`acc` de accuracy) durante el entrenamiento para evaluar el rendimiento.

**Propósito:** Configurar el proceso de entrenamiento del modelo.

**Importancia:** Sin estos componentes, el modelo no sabría cómo aprender de los datos ni cómo medir su propio rendimiento.

**Relación:** La compilación prepara el modelo `model` definido previamente para el proceso de `fit_generator` (entrenamiento).

In [8]:
from tensorflow.keras.optimizers import RMSprop

# Compila el modelo con la función de pérdida, el optimizador y las métricas especificadas
model.compile(loss='binary_crossentropy',
              optimizer=RMSprop(learning_rate=0.001),
              metrics=['acc'])

## Preprocesamiento de Datos con ImageDataGenerator

Para alimentar las imágenes a la red neuronal de manera eficiente y realizar algunas transformaciones básicas, se utilizan `ImageDataGenerator`s. Estos generadores leerán las imágenes de los directorios, las escalarán y las proporcionarán en lotes al modelo.

-   **`rescale=1./255`**: Normaliza los valores de píxeles de las imágenes del rango `[0, 255]` al rango `[0, 1]`, lo cual es una práctica estándar para las redes neuronales.
-   **`flow_from_directory(directorio, target_size, batch_size, class_mode)`**: Esta función lee las imágenes desde el `directorio` especificado, las redimensiona a `target_size` (150x150), las agrupa en `batch_size` (20) y asigna etiquetas binarias (`class_mode='binary'`) basándose en la estructura de subdirectorios.

**Propósito:** Preparar y alimentar las imágenes al modelo de manera estandarizada y en lotes.

**Importancia:** La normalización de píxeles es crucial para el rendimiento de la red, y los generadores de datos son eficientes para manejar grandes conjuntos de imágenes que no caben completamente en la memoria.

**Relación:** `train_generator` y `validation_generator` son las entradas directas al método `model.fit_generator` durante el entrenamiento.

In [13]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Crea un generador de datos para el entrenamiento que reescala los valores de los píxeles a [0, 1]
train_datagen = ImageDataGenerator(rescale=1./255)

# Crea un generador de datos para la validación que también reescala los valores de los píxeles
val_datagen = ImageDataGenerator(rescale=1./255)

# Configura el generador de imágenes de entrenamiento
train_generator = train_datagen.flow_from_directory(
        train_dir,  # Directorio fuente de las imágenes de entrenamiento
        target_size=(150, 150),  # Todas las imágenes se redimensionarán a 150x150
        batch_size=20, # Las imágenes se entregarán en lotes de 20
        class_mode='binary') # Como usamos 'binary_crossentropy', necesitamos etiquetas binarias

# Configura el generador de imágenes de validación
validation_generator = val_datagen.flow_from_directory(
        validation_dir, # Directorio fuente de las imágenes de validación
        target_size=(150, 150),
        batch_size=20,
        class_mode='binary')

NameError: name 'train_dir' is not defined

## Entrenamiento del Modelo

En esta sección, se entrena el modelo utilizando el método `fit_generator`. Este método es ideal cuando se trabaja con `ImageDataGenerator`s que proporcionan lotes de datos bajo demanda.

-   **`train_generator`**: Fuente de datos de entrenamiento.
-   **`steps_per_epoch`**: Número de lotes que se tomarán del generador de entrenamiento por época. Se calcula como `total_imagenes_entrenamiento / batch_size` (2000 / 20 = 100).
-   **`epochs`**: Número de veces que el modelo iterará sobre todo el conjunto de datos de entrenamiento (15 veces).
-   **`validation_data`**: Fuente de datos de validación.
-   **`validation_steps`**: Número de lotes que se tomarán del generador de validación por época (1000 / 20 = 50).
-   **`verbose=2`**: Muestra una barra de progreso por época, pero menos detallada que `verbose=1`.

**Propósito:** Ajustar los pesos del modelo para que aprenda a clasificar imágenes de perros y gatos.

**Importancia:** Es el paso central donde el modelo aprende de los datos.

**Relación:** Este proceso utiliza el `model` compilado y los `train_generator` y `validation_generator` definidos anteriormente. Los resultados del entrenamiento se almacenan en el objeto `history` para su posterior visualización.

In [14]:
# Entrena el modelo utilizando los generadores de datos
history = model.fit(
      train_generator,
      steps_per_epoch=100,  # 2000 imágenes / 20 batch_size = 100 pasos por época
      epochs=15, # Entrena durante 15 épocas
      validation_data=validation_generator,
      validation_steps=50,  # 1000 imágenes / 20 batch_size = 50 pasos de validación
      verbose=2)

NameError: name 'train_generator' is not defined

## Visualización de Representaciones Intermedias

Esta sección permite "ver" lo que las diferentes capas de la red neuronal están aprendiendo. Se crea un modelo de visualización que toma una imagen de entrada y devuelve las salidas de cada capa convolucional.

-   **`visualization_model = Model(img_input, successive_outputs)`**: Crea un nuevo modelo que mapea la entrada a las salidas de las capas intermedias.
-   **`img_to_array`, `load_img`**: Carga y preprocesa una imagen aleatoria (normalizando píxeles de 0-255 a 0-1).
-   El bucle itera sobre las salidas de las capas (`feature_map`) y las muestra, normalizando cada canal para una mejor visualización. Esto revela cómo las características se vuelven más abstractas y especializadas en cada capa.

**Propósito:** Entender qué tipos de características está aprendiendo la red en cada etapa.

**Importancia:** Es una herramienta de depuración y comprensión. Muestra cómo los píxeles brutos se transforman en representaciones de alto nivel, y cómo la red enfoca su atención.

**Relación:** Utiliza el `model` ya entrenado para observar su comportamiento interno con una imagen de ejemplo. Las capas `Conv2D` y `MaxPooling2D` son las que producen estos mapas de características.

In [11]:
import numpy as np
import random
from tensorflow.keras.preprocessing.image import img_to_array, load_img

# Define un nuevo modelo que tomará una imagen como entrada y devolverá las representaciones
# intermedias de todas las capas del modelo original (excluyendo la capa de entrada).
successive_outputs = [layer.output for layer in model.layers[1:]]
visualization_model = Model(img_input, successive_outputs)

# Prepara una imagen aleatoria (gato o perro) del conjunto de entrenamiento
cat_img_files = [os.path.join(train_cats_dir, f) for f in train_cat_fnames]
dog_img_files = [os.path.join(train_dogs_dir, f) for f in train_dog_fnames]
img_path = random.choice(cat_img_files + dog_img_files)

# Carga la imagen, la redimensiona y la convierte a un array NumPy
img = load_img(img_path, target_size=(150, 150))  # Carga como imagen PIL
x = img_to_array(img)  # Convierte a array NumPy (150, 150, 3)
x = x.reshape((1,) + x.shape)  # Añade una dimensión para el batch (1, 150, 150, 3)

# Re-escala los valores de píxeles a [0, 1]
x /= 255

# Ejecuta la imagen a través del modelo de visualización para obtener todas las representaciones intermedias
successive_feature_maps = visualization_model.predict(x)

# Obtiene los nombres de las capas para las etiquetas de las gráficas
layer_names = [layer.name for layer in model.layers[1:]]

# Itera sobre los mapas de características y los muestra
for layer_name, feature_map in zip(layer_names, successive_feature_maps):
  # Solo muestra las capas convolucionales y de max-pooling
  if len(feature_map.shape) == 4:
    n_features = feature_map.shape[-1]  # Número de características (filtros) en el mapa
    size = feature_map.shape[1]  # Tamaño del mapa de características (ej. 148x148)

    # Crea una cuadrícula para mostrar todos los filtros de la capa
    display_grid = np.zeros((size, size * n_features))
    for i in range(n_features):
      # Post-procesa cada filtro para hacerlo visualmente agradable
      x = feature_map[0, :, :, i]
      x -= x.mean()
      x /= x.std()
      x *= 64
      x += 128
      x = np.clip(x, 0, 255).astype('uint8')
      # Coloca el filtro en la cuadrícula
      display_grid[:, i * size : (i + 1) * size] = x

    # Muestra la cuadrícula de filtros
    scale = 20. / n_features
    plt.figure(figsize=(scale * n_features, scale))
    plt.title(layer_name)
    plt.grid(False)
    plt.imshow(display_grid, aspect='auto', cmap='viridis')
plt.show()

NameError: name 'train_cat_fnames' is not defined

## Evaluación de la Precisión y Pérdida del Modelo

Esta sección se centra en visualizar el rendimiento del modelo durante el entrenamiento. Se utilizan los datos registrados en el objeto `history` (retornado por `model.fit_generator`) para graficar la precisión y la pérdida tanto para el conjunto de entrenamiento como para el de validación.

-   **`history.history['acc']`**, **`history.history['val_acc']`**: Contienen los valores de precisión para el entrenamiento y la validación, respectivamente, en cada época.
-   **`history.history['loss']`**, **`history.history['val_loss']`**: Contienen los valores de la función de pérdida para el entrenamiento y la validación, respectivamente, en cada época.
-   **`plt.plot()`**: Se usa para dibujar las curvas de precisión y pérdida a lo largo de las épocas.

**Propósito:** Monitorear el progreso del entrenamiento y detectar problemas como el sobreajuste (overfitting).

**Importancia:** Estas gráficas son cruciales para entender si el modelo está aprendiendo bien, si se está generalizando a datos no vistos y si necesita ajustes (por ejemplo, más datos, regularización).

**Relación:** Los datos para estas gráficas provienen directamente del proceso de entrenamiento iniciado por `model.fit_generator`. Las conclusiones obtenidas de estas gráficas guían las próximas etapas de mejora del modelo.

In [12]:
import matplotlib.pyplot as plt

# Recupera las listas de resultados de precisión y pérdida para entrenamiento y validación por época
acc = history.history['acc']
val_acc = history.history['val_acc']

loss = history.history['loss']
val_loss = history.history['val_loss']

# Obtiene el número de épocas
epochs = range(len(acc))

# Grafica la precisión de entrenamiento y validación por época
plt.plot(epochs, acc, label='Precisión de Entrenamiento')
plt.plot(epochs, val_acc, label='Precisión de Validación')
plt.title('Precisión de Entrenamiento y Validación')
plt.legend()

plt.figure() # Crea una nueva figura para la gráfica de pérdida

# Grafica la pérdida de entrenamiento y validación por época
plt.plot(epochs, loss, label='Pérdida de Entrenamiento')
plt.plot(epochs, val_loss, label='Pérdida de Validación')
plt.title('Pérdida de Entrenamiento y Validación')
plt.legend()
plt.show()

NameError: name 'history' is not defined

## Diagnóstico de Sobreajuste (Overfitting)

Las gráficas de precisión y pérdida del paso anterior revelan un problema común en el aprendizaje automático: el sobreajuste. Esto ocurre cuando un modelo aprende los datos de entrenamiento demasiado bien, memorizando ruido y patrones específicos que no se generalizan a datos nuevos.

-   **Precisión de entrenamiento (azul):** Se acerca al 100%, indicando que el modelo se desempeñaba muy bien con los datos que ya había visto.
-   **Precisión de validación (verde):** Se estanca alrededor del 70%, lo que significa que el modelo no puede generalizar su conocimiento a imágenes nuevas.
-   **Pérdida de validación:** Alcanza su mínimo temprano y luego comienza a aumentar, mientras que la pérdida de entrenamiento sigue disminuyendo. Esto es una señal clara de sobreajuste.

**Propósito:** Identificar el sobreajuste como el problema principal con el modelo actual.

**Importancia:** El sobreajuste es un obstáculo crítico para construir modelos útiles. Comprenderlo es el primer paso para aplicar técnicas que lo mitiguen y mejorar la capacidad de generalización del modelo.

**Relación:** Esta sección interpreta los resultados de `model.fit_generator` y las visualizaciones de las gráficas de `history`, estableciendo el contexto para las futuras mejoras que se abordarían en un ejercicio posterior.

## Limpieza y Liberación de Recursos

La última celda del cuaderno tiene la función de limpiar el entorno y liberar los recursos de memoria. Esto es una buena práctica, especialmente en entornos como Google Colab, donde los recursos de computación son compartidos y limitados.

-   **`os.kill(os.getpid(), signal.SIGKILL)`**: Este comando termina el proceso actual del kernel de Python, liberando toda la memoria y recursos asociados a la ejecución del cuaderno.

**Propósito:** Reiniciar el entorno de ejecución del cuaderno.

**Importancia:** Evita conflictos de memoria o recursos con ejecuciones posteriores y asegura que el siguiente ejercicio comience con un estado limpio.

**Relación:** Es una acción final para cerrar el ciclo de este ejercicio antes de comenzar con el siguiente, que probablemente construya sobre este trabajo.

In [ ]:
import os, signal
os.kill(os.getpid(), signal.SIGKILL)